# Compare Ecowitt data AW005 with reference sensors from DGA, Chile

This notebook is designed for **Google Colab**.

It does four things:

1. Clones or reads your repository.
2. Loads **Ecowitt** data from the repo.
3. Loads the **reference sensor** file from the repo.
4. Aligns timestamps, compares selected variables, computes metrics, and creates plots.

In [2]:
#@title 1) Install / import packages
import os
import subprocess
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Configuration

Edit the values below before running.

Notes:

- `REPO_URL` should point to your repository.
- `ECOWITT_FILE` and `REFERENCE_FILE` must be paths **inside the repo**.
- `VARIABLE_MAP` links the Ecowitt column name to the matching reference column name.
- If your Ecowitt file has no header, the notebook can assign default fallback names.

In [3]:
#@title 2) User configuration

# --- Repository ---
REPO_URL = "https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027"   # change this
REPO_BRANCH = "main"                                   # change if needed
REPO_DIR = "/content/VLIR_SI_EC_2025-2027"              # change this

# Set to False if the repo is already available in /content
CLONE_REPO = True

# --- Files inside the repo ---
ECOWITT_FILE = "datasets/AW005.csv"                # This is for Chillán, Chile. Change this for other sensors
REFERENCE_FILE = "Validation_meteo/DGA_ChillanUdeC.xlsx" # Same here, change the reference file

# Time columns
ECOWITT_TIME_COL = "timestamp"
REFERENCE_TIME_COL = "Timestamp"   # change to the real reference time column

# Set timezone if needed, for example "Europe/Brussels"
TIMEZONE = None

# Common comparison timestep
RESAMPLE_FREQ = "1H"   # examples: "5min", "15min", "1H"

# Variable mapping
# Change only the "reference" names to your real column names
VARIABLE_MAP = {
    "temp_c": {
        "ecowitt": "outdoor_temp",
        "reference": "temp_ref"
    },
    "humidity_pct": {
        "ecowitt": "outdoor_humidity",
        "reference": "rh_ref"
    },
    "dew_point_c": {
        "ecowitt": "dew_point",
        "reference": "dew_point_ref"
    },
    "feels_like_c": {
        "ecowitt": "feels_like",
        "reference": "feels_like_ref"
    },
    "wind_speed": {
        "ecowitt": "wind_speed",
        "reference": "wind_speed_ref"
    },
    "wind_gust": {
        "ecowitt": "wind_gust",
        "reference": "wind_gust_ref"
    },
    "pressure": {
        "ecowitt": "pressure",
        "reference": "pressure_ref"
    },
    "rain_rate": {
        "ecowitt": "rain_rate",
        "reference": "rain_rate_ref"
    },
    "daily_rain": {
        "ecowitt": "daily_rain",
        "reference": "daily_rain_ref"
    },
    "solar_radiation": {
        "ecowitt": "solar_radiation",
        "reference": "solar_ref"
    },
    "uv_index": {
        "ecowitt": "uv_index",
        "reference": "uv_ref"
    },
}


In [7]:
#@title 3) Clone repo (skip if not needed)

if CLONE_REPO:
    if os.path.exists(REPO_DIR):
        print(f"Repository folder already exists: {REPO_DIR}")
    else:
        cmd = ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR]
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print(f"Using existing repo at: {REPO_DIR}")

print("Repo dir:", REPO_DIR)

Running: git clone --branch main https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027 /content/VLIR_SI_EC_2025-2027
Repo dir: /content/VLIR_SI_EC_2025-2027


In [8]:
#@title 4) Helper functions

DEFAULT_ECOWITT_COLUMNS_15 = [
    "timestamp",
    "outdoor_temp",
    "outdoor_humidity",
    "dew_point",
    "feels_like",
    "wind_speed",
    "wind_gust",
    "wind_dir",
    "pressure",
    "rain_rate",
    "daily_rain",
    "solar_radiation",
    "uv_index",
    "indoor_temp",
    "indoor_humidity",
]


def robust_read_csv(path):
    """
    Reads a CSV and tries to detect whether it has a header.
    If the first field looks like a timestamp and there are exactly 15 columns,
    fallback Ecowitt names are assigned.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    # First try normal header parsing
    df = pd.read_csv(path)

    # If the columns themselves look like data, reload without header
    first_col_name = str(df.columns[0])
    first_col_looks_like_time = pd.to_datetime(pd.Series([first_col_name]), errors="coerce").notna().iloc[0]

    if first_col_looks_like_time:
        df = pd.read_csv(path, header=None)
        if df.shape[1] == 15:
            df.columns = DEFAULT_ECOWITT_COLUMNS_15
        else:
            df.columns = [f"col_{i}" for i in range(df.shape[1])]

    # Clean BOM on first column if present
    df.columns = [str(c).replace("\\ufeff", "").strip() for c in df.columns]
    return df


def standardize_time(df, time_col, timezone=None):
    df = df.copy()
    if time_col not in df.columns:
        raise KeyError(f"Time column '{time_col}' not found. Available columns: {list(df.columns)}")

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    df = df.dropna(subset=[time_col])

    if timezone is not None:
        df[time_col] = df[time_col].dt.tz_convert(timezone)

    df = df.sort_values(time_col).drop_duplicates(subset=[time_col])
    return df


def apply_date_filter(df, time_col, start_date=None, end_date=None):
    df = df.copy()
    if start_date is not None:
        df = df[df[time_col] >= pd.to_datetime(start_date, utc=True)]
    if end_date is not None:
        df = df[df[time_col] <= pd.to_datetime(end_date, utc=True)]
    return df


def convert_numeric_columns(df, exclude=None):
    df = df.copy()
    exclude = set(exclude or [])
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors="ignore")
    return df


def resample_dataset(df, time_col, column_map, freq, agg_rules, dataset_name):
    """
    column_map is a dict like:
    {"temp": "outdoor_temp", "humidity": "outdoor_humidity"}
    """
    keep_cols = [time_col] + [c for c in column_map.values() if c in df.columns]
    tmp = df[keep_cols].copy()
    tmp = tmp.set_index(time_col)

    agg = {}
    rename_back = {}
    for logical_name, col in column_map.items():
        if col not in tmp.columns:
            print(f"[{dataset_name}] Missing column for '{logical_name}': {col}")
            continue
        agg[col] = agg_rules.get(logical_name, "mean")
        rename_back[col] = logical_name

    out = tmp.resample(freq).agg(agg)
    out = out.rename(columns=rename_back)
    out.index.name = time_col
    return out.reset_index()


def compute_metrics(y_true, y_pred):
    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "n": 0,
            "bias": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "corr": np.nan,
        }

    bias = float((y_pred - y_true).mean())
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else np.nan

    return {
        "n": int(len(y_true)),
        "bias": bias,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "corr": corr,
    }


def plot_timeseries(df, var, outdir):
    plot_df = df[["timestamp", f"{var}_ecowitt", f"{var}_reference"]].dropna(how="all")
    if plot_df.empty:
        print(f"No data to plot for {var}")
        return

    plt.figure(figsize=(14, 5))
    plt.plot(plot_df["timestamp"], plot_df[f"{var}_ecowitt"], label="Ecowitt")
    plt.plot(plot_df["timestamp"], plot_df[f"{var}_reference"], label="Reference")
    plt.title(f"Time series comparison: {var}")
    plt.xlabel("Time")
    plt.ylabel(var)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"timeseries_{var}.png"), dpi=150)
    plt.show()


def plot_scatter(df, var, outdir):
    plot_df = df[[f"{var}_reference", f"{var}_ecowitt"]].dropna()
    if plot_df.empty:
        print(f"No data to plot for {var}")
        return

    x = plot_df[f"{var}_reference"]
    y = plot_df[f"{var}_ecowitt"]

    plt.figure(figsize=(6, 6))
    plt.scatter(x, y, alpha=0.6)
    min_v = np.nanmin([x.min(), y.min()])
    max_v = np.nanmax([x.max(), y.max()])
    plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
    plt.title(f"Scatter comparison: {var}")
    plt.xlabel("Reference")
    plt.ylabel("Ecowitt")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"scatter_{var}.png"), dpi=150)
    plt.show()



def plot_single_series(df, time_col, value_col, title, ylabel):
    tmp = df[[time_col, value_col]].dropna()
    if tmp.empty:
        print(f"Skipping {value_col}: no data")
        return

    plt.figure(figsize=(12, 5))
    plt.plot(tmp[time_col], tmp[value_col])
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    #plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_comparison_timeseries(df, time_col, eco_col, ref_col, logical_var, outpath):
    tmp = df[[time_col, eco_col, ref_col]].dropna()
    if tmp.empty:
        print(f"Skipping comparison timeseries for {logical_var}: no overlapping data")
        return

    plt.figure(figsize=(12, 5))
    plt.plot(tmp[time_col], tmp[eco_col], label="Ecowitt")
    plt.plot(tmp[time_col], tmp[ref_col], label="Reference")
    plt.title(f"Comparison time series: {logical_var}")
    plt.xlabel("Time")
    plt.ylabel(logical_var)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_scatter_comparison(df, eco_col, ref_col, logical_var, outpath):
    tmp = df[[eco_col, ref_col]].dropna()
    if tmp.empty:
        print(f"Skipping scatter for {logical_var}: no overlapping data")
        return

    plt.figure(figsize=(6, 6))
    plt.scatter(tmp[ref_col], tmp[eco_col], alpha=0.6)

    mn = min(tmp[ref_col].min(), tmp[eco_col].min())
    mx = max(tmp[ref_col].max(), tmp[eco_col].max())
    plt.plot([mn, mx], [mn, mx])

    plt.xlabel("Reference")
    plt.ylabel("Ecowitt")
    plt.title(f"Scatter comparison: {logical_var}")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

# =========================
# Aggregation rules for resampling
# =========================
AGG_RULES = {
    "outdoor_temp": "mean",
    "outdoor_humidity": "mean",
    "dew_point": "mean",
    "feels_like": "mean",
    "wind_speed": "mean",
    "wind_gust": "max",
    "pressure": "mean",
    "rain_rate": "mean",
    "daily_rain": "max",
    "solar_radiation": "mean",
    "uv_index": "max",
}

In [43]:
#@title 5) Load ecowitt data from repo

ecowitt_path = os.path.join(REPO_DIR, ECOWITT_FILE)

# Load raw CSV
ecowitt_hourly = robust_read_csv(ecowitt_path)

# Rename columns for consistency
ecowitt_hourly = ecowitt_hourly.rename(columns={

    "timestamp": "Date",

    "outdoor_temp": "air_temperature_c",

    "outdoor_humidity": "relative_humidity_pct",

    "dew_point": "dew_point_c",

    "feels_like": "feels_like_c",

    "wind_speed": "wind_speed_ms",

    "wind_gust": "wind_gust_ms",

    "wind_dir": "wind_direction_deg",

    "pressure": "atmospheric_pressure_hpa",

    "rain_rate": "rain_rate_mm_hr",

    "daily_rain": "daily_rain_mm",

    "solar_radiation": "solar_radiation_w_m2",

    "uv_index": "uv_index",

    "indoor_temp": "indoor_temperature_c",

    "indoor_humidity": "indoor_relative_humidity_pct"

})

print("Ecowitt shape:", ecowitt_hourly.shape)

display(ecowitt_hourly.head())

Ecowitt shape: (3375, 15)


,Date,air_temperature_c,relative_humidity_pct,dew_point_c,feels_like_c,wind_speed_ms,wind_gust_ms,wind_direction_deg,atmospheric_pressure_hpa,rain_rate_mm_hr,daily_rain_mm,solar_radiation_w_m2,uv_index,indoor_temperature_c,indoor_relative_humidity_pct
0,2025-12-22T18:01:17.609Z,26.0,41.0,11.7,26.0,0.0,0.0,202.0,29.92,0.0,0.0,2.4,0.0,24.9,42.0
1,2025-12-22T19:01:17.852Z,26.3,40.0,11.6,26.3,0.0,0.0,127.0,29.91,0.0,0.8,2.1,0.0,26.1,40.0
2,2025-12-22T20:01:17.232Z,26.9,36.0,10.6,26.6,0.0,0.0,245.0,29.90,0.0,0.3,1.9,0.0,26.7,37.0
3,2025-12-22T21:01:18.421Z,26.8,36.0,10.5,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.7,0.0,26.4,36.0
4,2025-12-22T22:01:14.613Z,26.5,35.0,9.8,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.4,0.0,26.1,35.0


In [44]:
#@title 6) Load and resample reference data

reference_path = os.path.join(REPO_DIR, REFERENCE_FILE)

# Read Excel using second row as header
reference_raw = pd.read_excel(
    reference_path,
    header=1
)

# Rename datetime column
reference_raw = reference_raw.rename(columns={
    "Unnamed: 1": "Date"
})

# Remove unwanted column
reference_raw = reference_raw.drop(columns=["Unnamed: 0"], errors="ignore")

# Convert Date to datetime
reference_raw["Date"] = pd.to_datetime(
    reference_raw["Date"],
    errors="coerce"
)

# Remove invalid dates
reference_raw = reference_raw.dropna(subset=["Date"])

# Set datetime index
reference_raw = reference_raw.set_index("Date")

# Rename columns to English
reference_raw = reference_raw.rename(columns={

    "Pptación Acum. (mm)": "precipitation_mm",

    "Temp.del Aire (ºC)": "air_temperature_c",

    "Humedad (%)": "relative_humidity_pct",

    "Rad.Solar (Watt/m2)": "solar_radiation_w_m2",

    "Presión Atmosférica (mb)": "atmospheric_pressure_mb",

    "Pp. acum. disdrometro (mm)": "disdrometer_precipitation_mm",

    "Direccion del Viento (grados)": "wind_direction_deg",

    "Velocidad del Vto. (m/s)": "wind_speed_ms",

    "Calidad de señal (db)": "signal_quality_db",

    "Pp. instan. disdrometro (mm)": "disdrometer_instantaneous_rainfall_mm",

    "Tipo Pptacion disdrometro ( )": "disdrometer_precipitation_type"

})

# Sort chronologically
reference_raw = reference_raw.sort_index()

# Convert numeric columns
for col in reference_raw.columns:
    if col != "disdrometer_precipitation_type":
        reference_raw[col] = pd.to_numeric(reference_raw[col], errors="coerce")

# Aggregation rules
agg_rules = {}

for col in reference_raw.columns:

    # precipitation columns → sum
    if col in [
        "precipitation_mm",
        "disdrometer_precipitation_mm",
        "disdrometer_instantaneous_rainfall_mm"
    ]:
        agg_rules[col] = "sum"

    # precipitation type → last value
    elif col == "disdrometer_precipitation_type":
        agg_rules[col] = "last"

    # everything else → mean
    else:
        agg_rules[col] = "mean"

# Resample hourly
reference_hourly = reference_raw.resample("1H").agg(agg_rules)

# Remove fully empty hours
reference_hourly = reference_hourly.dropna(how="all")

print("Original reference shape:", reference_raw.shape)
print("Hourly reference shape:", reference_hourly.shape)

display(reference_hourly.head())

Original reference shape: (4310, 11)
Hourly reference shape: (2157, 11)


/tmp/ipykernel_674/3202656188.py:88: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  reference_hourly = reference_raw.resample("1H").agg(agg_rules)


,precipitation_mm,air_temperature_c,relative_humidity_pct,solar_radiation_w_m2,atmospheric_pressure_mb,disdrometer_precipitation_mm,wind_direction_deg,wind_speed_ms,signal_quality_db,disdrometer_instantaneous_rainfall_mm,disdrometer_precipitation_type
Date,,,,,,,,,,,
2026-02-13 00:00:00,11.4,17.70,39.80,0.0,1002.55,1240.6,200.50,2.1,-113.0,0.0,0
2026-02-13 01:00:00,11.4,16.35,47.20,0.0,1002.35,1240.6,225.45,1.8,-113.0,0.0,0
2026-02-13 02:00:00,11.4,15.25,53.30,0.0,1002.15,1240.6,221.05,0.8,-113.0,0.0,0
2026-02-13 03:00:00,11.4,14.55,57.80,0.0,1001.75,1240.6,216.25,1.4,-113.0,0.0,0
2026-02-13 04:00:00,11.4,14.05,63.15,0.0,1001.50,1240.6,194.25,1.0,-113.0,0.0,0


In [45]:
#@title 8) Resample to common timestep

for logical_var in VARIABLE_MAP.keys():

    if logical_var in ecowitt.columns:
        plot_single_series(
            ecowitt,
            time_col="time",
            value_col=logical_var,
            title=f"Ecowitt: {logical_var}",
            ylabel=logical_var,
        )
    else:
        print(f"Skipping {logical_var}: column not found")



Skipping temp_c: column not found
Skipping humidity_pct: column not found


KeyError: "['time'] not in index"